# SASRec Stage 3 Baseline Multi-Task BPI2012 Colab Train 01

This notebook runs the first Stage 3 baseline multi-task experiments.

Initial recommendation:
- run `s42` first
- only add `s2024` and `s7` if the first results look meaningful

Outputs:
- next activity
- next time

Best epoch criterion:
- `full_valid_ndcg@10`


In [1]:
import torch

print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu name:', torch.cuda.get_device_name(0))


torch version: 2.11.0+cu128
cuda available: True
gpu name: Tesla T4


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
GITHUB_USERNAME = 'hwbuzz'

DRIVE_ROOT = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction'
REPO_DIR = '/content/time-aware-behavior-prediction'

DATA_DIR = f'{DRIVE_ROOT}/data/processed/bpi2012_complete_only'
BASELINE_NDCG10_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10'
MULTITASK_BASELINE_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_baseline_multitask_ndcg10'
NOTEBOOK_DIR = f'{DRIVE_ROOT}/notebooks'

print('DATA_DIR:', DATA_DIR)
print('BASELINE_NDCG10_OUTPUT_DIR:', BASELINE_NDCG10_OUTPUT_DIR)
print('MULTITASK_BASELINE_OUTPUT_DIR:', MULTITASK_BASELINE_OUTPUT_DIR)
print('NOTEBOOK_DIR:', NOTEBOOK_DIR)


DATA_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/data/processed/bpi2012_complete_only
BASELINE_NDCG10_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10
MULTITASK_BASELINE_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_baseline_multitask_ndcg10
NOTEBOOK_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/notebooks


In [4]:
!mkdir -p "$NOTEBOOK_DIR"
!mkdir -p "$DATA_DIR"
!mkdir -p "$BASELINE_NDCG10_OUTPUT_DIR"
!mkdir -p "$MULTITASK_BASELINE_OUTPUT_DIR"


In [5]:
%cd /content
!test -d time-aware-behavior-prediction || git clone https://github.com/$GITHUB_USERNAME/time-aware-behavior-prediction.git
%cd /content/time-aware-behavior-prediction
!git pull


/content
/content/time-aware-behavior-prediction
Already up to date.


In [6]:
%cd /content/time-aware-behavior-prediction

skip_packages = ['pywinpty']

with open('requirements.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()

with open('requirements_colab.txt', 'w', encoding='utf-8') as f:
    for line in lines:
        pkg = line.strip().lower()
        if not any(name in pkg for name in skip_packages):
            f.write(line)

print('created requirements_colab.txt')


/content/time-aware-behavior-prediction
created requirements_colab.txt


In [7]:
!pip install -r requirements_colab.txt


In [8]:
!ls "$DATA_DIR"


events_complete_only_filtered.csv  sasrec_interactions.csv
events_encoded_time_features.csv   sasrec_interactions.txt
item_map.csv			   user_map.csv


In [9]:
%cd /content/time-aware-behavior-prediction
!ls data/processed/bpi2012_complete_only


/content/time-aware-behavior-prediction
events_complete_only_filtered.csv  sasrec_interactions.csv
events_encoded_time_features.csv   sasrec_interactions.txt
item_map.csv			   user_map.csv


## Verify Stage 3 processed file

Stage 3 next-time prediction uses the official processed time-feature CSV.
This check confirms that `delta_next_seconds` already exists before training.


In [10]:
import pandas as pd

time_features_path = 'data/processed/bpi2012_complete_only/events_encoded_time_features.csv'
df = pd.read_csv(time_features_path)
required_cols = [
    'delta_prev_seconds',
    'delta_start_seconds',
    'delta_next_seconds',
]
missing = [c for c in required_cols if c not in df.columns]

if missing:
    raise ValueError(f'Missing required Stage 3 columns: {missing}')

print('Stage 3 processed file is ready.')
print(df.columns.tolist())
df[['user_id', 'event_idx', 'delta_prev_seconds', 'delta_start_seconds', 'delta_next_seconds']].head()


Stage 3 processed file is ready.
['case_id', 'activity', 'lifecycle', 'timestamp', 'event_idx', 'delta_prev_seconds', 'delta_start_seconds', 'delta_next_seconds', 'user_id', 'item_id']


,user_id,event_idx,delta_prev_seconds,delta_start_seconds,delta_next_seconds
0,1,0,0.000,0.000,0.334
1,1,1,0.334,0.334,53.026
2,1,2,53.026,53.360,39785.402
3,1,3,39785.402,39838.762,145.935
4,1,4,145.935,39984.697,-0.000


## Experiment design

Stage 3 baseline multi-task runs:

- backbone 1: `anchor_ml20`
  - `hidden_units=50, num_blocks=2, num_heads=1, maxlen=20, lr=0.001, dropout=0.2`
- backbone 2: `refine_ml50_do035`
  - `hidden_units=50, num_blocks=2, num_heads=1, maxlen=50, lr=0.001, dropout=0.35`
- first run: `seed=42`
- additional seeds only if needed: `2024`, `7`
- multi-task output: `next activity + next time`
- time target: `delta_next_seconds`
- time target transform: `log1p`
- time loss: `huber`
- time loss weight: `1.0`
- best epoch criterion: `full_valid_ndcg@10`


## Check prerequisite baseline runs


In [11]:
from pathlib import Path

checks = [
    ('Baseline NDCG@10', Path(BASELINE_NDCG10_OUTPUT_DIR), [
        'anchor_ml20_s42',
        'refine_ml50_do035_s42',
    ]),
]

for label, output_dir, run_names in checks:
    print('=' * 80)
    print(label)
    for run_name in run_names:
        run_dir = output_dir / run_name
        print(run_name, 'EXISTS' if run_dir.exists() else 'MISSING')


Baseline NDCG@10
anchor_ml20_s42 EXISTS
refine_ml50_do035_s42 EXISTS


## Check planned initial runs


In [12]:
planned_runs = [
    'multitask_anchor_ml20_s42',
    'multitask_refine_ml50_do035_s42',
]

output_dir = Path(MULTITASK_BASELINE_OUTPUT_DIR)
print('=' * 80)
print('Stage 3 baseline multi-task initial runs')
for run_name in planned_runs:
    run_dir = output_dir / run_name
    print(run_name, 'EXISTS' if run_dir.exists() else 'OK')


Stage 3 baseline multi-task initial runs
multitask_anchor_ml20_s42 OK
multitask_refine_ml50_do035_s42 OK


## Train initial baseline multi-task runs (`s42` first)


### multitask_anchor_ml20_s42


In [13]:
!python src/train_sasrec.py \
  --run_name multitask_anchor_ml20_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 42 \
  --enable_time_prediction \
  --time_prediction_target delta_next_seconds \
  --time_target_transform log1p \
  --time_loss_type huber \
  --time_loss_weight 1.0 \
  --output_dir "$MULTITASK_BASELINE_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --time_features_path data/processed/bpi2012_complete_only/events_encoded_time_features.csv \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_baseline_multitask_ndcg10/multitask_anchor_ml20_s42
epoch=1, loss=2.5763
epoch=2, loss=1.5438
epoch=3, loss=1.4283
epoch=4, loss=1.3598
epoch=5, loss=1.3169
valid [task], Top5Acc: 0.4004, Top10Acc: 0.7708, Acc: 0.0551, MacroF1: 0.0988, TimeMAE: 70257.4629, TimeRMSE: 275813.0764, TimeMedAE: 878.5276
valid [full], NDCG@5: 0.6264, HR@5: 0.7599, NDCG@10: 0.6882, HR@10: 0.9589, MRR: 0.6119
valid [sampled], NDCG@5: 0.5231, HR@5: 0.5234, NDCG@10: 0.5258, HR@10: 0.5317, MRR: 0.5389
test [task], Top5Acc: 0.0465, Top10Acc: 0.5384, Acc: 0.0141, MacroF1: 0.0150, TimeMAE: 11838.0120, Ti

### multitask_refine_ml50_do035_s42


In [14]:
!python src/train_sasrec.py \
  --run_name multitask_refine_ml50_do035_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 42 \
  --enable_time_prediction \
  --time_prediction_target delta_next_seconds \
  --time_target_transform log1p \
  --time_loss_type huber \
  --time_loss_weight 1.0 \
  --output_dir "$MULTITASK_BASELINE_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --time_features_path data/processed/bpi2012_complete_only/events_encoded_time_features.csv \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_baseline_multitask_ndcg10/multitask_refine_ml50_do035_s42
epoch=1, loss=2.8505
epoch=2, loss=1.7178
epoch=3, loss=1.5850
epoch=4, loss=1.4952
epoch=5, loss=1.4477
valid [task], Top5Acc: 0.3364, Top10Acc: 0.7286, Acc: 0.0636, MacroF1: 0.0966, TimeMAE: 68934.1766, TimeRMSE: 276896.8925, TimeMedAE: 684.3376
valid [full], NDCG@5: 0.5971, HR@5: 0.7148, NDCG@10: 0.6821, HR@10: 0.9764, MRR: 0.5973
valid [sampled], NDCG@5: 0.5073, HR@5: 0.5086, NDCG@10: 0.5088, HR@10: 0.5135, MRR: 0.5222
test [task], Top5Acc: 0.0402, Top10Acc: 0.4269, Acc: 0.0252, MacroF1: 0.0224, TimeMAE: 11305.02

## Optional additional seeds (`s2024`, `s7`)

Run this section only if the initial `s42` results look meaningful.


### multitask_anchor_ml20_s2024


In [ ]:
!python src/train_sasrec.py \
  --run_name multitask_anchor_ml20_s2024 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 2024 \
  --enable_time_prediction \
  --time_prediction_target delta_next_seconds \
  --time_target_transform log1p \
  --time_loss_type huber \
  --time_loss_weight 1.0 \
  --output_dir "$MULTITASK_BASELINE_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --time_features_path data/processed/bpi2012_complete_only/events_encoded_time_features.csv \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


### multitask_anchor_ml20_s7


In [ ]:
!python src/train_sasrec.py \
  --run_name multitask_anchor_ml20_s7 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 7 \
  --enable_time_prediction \
  --time_prediction_target delta_next_seconds \
  --time_target_transform log1p \
  --time_loss_type huber \
  --time_loss_weight 1.0 \
  --output_dir "$MULTITASK_BASELINE_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --time_features_path data/processed/bpi2012_complete_only/events_encoded_time_features.csv \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


### multitask_refine_ml50_do035_s2024


In [ ]:
!python src/train_sasrec.py \
  --run_name multitask_refine_ml50_do035_s2024 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 2024 \
  --enable_time_prediction \
  --time_prediction_target delta_next_seconds \
  --time_target_transform log1p \
  --time_loss_type huber \
  --time_loss_weight 1.0 \
  --output_dir "$MULTITASK_BASELINE_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --time_features_path data/processed/bpi2012_complete_only/events_encoded_time_features.csv \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


### multitask_refine_ml50_do035_s7


In [ ]:
!python src/train_sasrec.py \
  --run_name multitask_refine_ml50_do035_s7 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 7 \
  --enable_time_prediction \
  --time_prediction_target delta_next_seconds \
  --time_target_transform log1p \
  --time_loss_type huber \
  --time_loss_weight 1.0 \
  --output_dir "$MULTITASK_BASELINE_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --time_features_path data/processed/bpi2012_complete_only/events_encoded_time_features.csv \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


## Rebuild result tables


In [15]:
from pathlib import Path
import json
import pandas as pd

def rebuild_df(output_dir: str):
    rows = []
    output_path = Path(output_dir)
    if not output_path.exists():
        return pd.DataFrame()
    for run_dir in output_path.iterdir():
        if not run_dir.is_dir():
            continue
        summary_path = run_dir / 'metrics_summary.json'
        config_path = run_dir / 'config.json'
        if not summary_path.exists() or not config_path.exists():
            continue
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        config = json.loads(config_path.read_text(encoding='utf-8'))
        row = {
            'run_name': summary.get('run_name'),
            'run_dir': str(run_dir),
            'completed_at': summary.get('completed_at'),
            'best_epoch': summary.get('best_epoch'),
            'checkpoint_best': summary.get('checkpoint_best'),
            'checkpoint_last': summary.get('checkpoint_last'),
            'metrics_history': summary.get('metrics_history'),
            'config_path': str(config_path),
            'metrics_summary': str(summary_path),
            'maxlen': config.get('maxlen'),
            'dropout_rate': config.get('dropout_rate'),
            'hidden_units': config.get('hidden_units'),
            'seed': config.get('seed'),
            'selection_metric': config.get('selection_metric'),
            'enable_time_prediction': config.get('enable_time_prediction', False),
            'time_prediction_target': config.get('time_prediction_target'),
            'time_loss_weight': config.get('time_loss_weight'),
            'time_target_transform': config.get('time_target_transform'),
            'time_modeling_mode': config.get('time_modeling_mode'),
        }
        for group_name in ['best_valid', 'best_test_at_best_valid', 'last_valid', 'last_test']:
            group = summary.get(group_name) or {}
            for mode, metrics in group.items():
                for key, value in metrics.items():
                    row[f'{group_name}_{mode}_{key}'] = value
        rows.append(row)
    return pd.DataFrame(rows)


In [16]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1200)
pd.set_option('display.max_colwidth', None)


## Comparison summary


In [17]:
baseline_runs = [
    'anchor_ml20_s42',
    'refine_ml50_do035_s42',
]
multitask_runs = [
    'multitask_anchor_ml20_s42',
    'multitask_refine_ml50_do035_s42',
]

baseline_df = rebuild_df(BASELINE_NDCG10_OUTPUT_DIR)
multitask_df = rebuild_df(MULTITASK_BASELINE_OUTPUT_DIR)

baseline_subset = baseline_df[baseline_df['run_name'].isin(baseline_runs)].copy()
baseline_subset['variant'] = baseline_subset['run_name'].map(lambda x: 'anchor_single_task' if x.startswith('anchor_ml20') else 'refine_single_task')

multitask_subset = multitask_df[multitask_df['run_name'].isin(multitask_runs)].copy()
multitask_subset['variant'] = multitask_subset['run_name'].map(lambda x: 'anchor_multi_task' if 'anchor_ml20' in x else 'refine_multi_task')

df_compare = pd.concat([baseline_subset, multitask_subset], ignore_index=True)
df_compare = df_compare.sort_values(['variant', 'seed', 'run_name']).reset_index(drop=True)

df_compare[[
    'run_name', 'seed', 'variant', 'maxlen', 'dropout_rate', 'selection_metric',
    'best_valid_full_ndcg@10', 'best_valid_full_hr@10', 'best_valid_full_mrr',
    'best_test_at_best_valid_full_ndcg@10', 'best_test_at_best_valid_full_hr@10', 'best_test_at_best_valid_full_mrr',
    'best_valid_accuracy', 'best_valid_macro_f1', 'best_valid_top5_accuracy', 'best_valid_top10_accuracy',
    'best_test_at_best_valid_accuracy', 'best_test_at_best_valid_macro_f1', 'best_test_at_best_valid_top5_accuracy', 'best_test_at_best_valid_top10_accuracy',
    'best_valid_time_mae', 'best_valid_time_rmse', 'best_valid_time_median_ae',
    'best_test_at_best_valid_time_mae', 'best_test_at_best_valid_time_rmse', 'best_test_at_best_valid_time_median_ae',
]]


,run_name,seed,variant,maxlen,dropout_rate,selection_metric,best_valid_full_ndcg@10,best_valid_full_hr@10,best_valid_full_mrr,best_test_at_best_valid_full_ndcg@10,best_test_at_best_valid_full_hr@10,best_test_at_best_valid_full_mrr,best_valid_accuracy,best_valid_macro_f1,best_valid_top5_accuracy,best_valid_top10_accuracy,best_test_at_best_valid_accuracy,best_test_at_best_valid_macro_f1,best_test_at_best_valid_top5_accuracy,best_test_at_best_valid_top10_accuracy,best_valid_time_mae,best_valid_time_rmse,best_valid_time_median_ae,best_test_at_best_valid_time_mae,best_test_at_best_valid_time_rmse,best_test_at_best_valid_time_median_ae
0,multitask_anchor_ml20_s42,42,anchor_multi_task,20,0.20,full_valid_ndcg@10,0.742361,0.976065,0.671685,0.809234,1.0,0.745731,0.053685,0.052223,0.408384,0.692765,0.014680,0.011972,0.478864,0.668207,74664.950418,282278.030256,6863.672687,14168.895027,78819.227077,129.408642
1,anchor_ml20_s42,42,anchor_single_task,20,0.20,full_valid_ndcg@10,0.750367,0.977546,0.681799,0.915296,1.0,0.886842,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,multitask_refine_ml50_do035_s42,42,refine_multi_task,50,0.35,full_valid_ndcg@10,0.733884,0.968635,0.668363,0.661596,1.0,0.551116,0.063544,0.086081,0.451324,0.704549,0.027067,0.020993,0.150494,0.519962,73264.328087,282009.630211,718.422925,11773.042436,74347.935455,49.665553
3,refine_ml50_do035_s42,42,refine_single_task,50,0.35,full_valid_ndcg@10,0.735378,0.977276,0.663510,0.891774,1.0,0.858375,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
summary_compare = df_compare.groupby('variant')[[
    'best_valid_full_ndcg@10', 'best_valid_full_hr@10', 'best_valid_full_mrr',
    'best_test_at_best_valid_full_ndcg@10', 'best_test_at_best_valid_full_hr@10', 'best_test_at_best_valid_full_mrr',
    'best_valid_accuracy', 'best_valid_macro_f1', 'best_valid_top5_accuracy', 'best_valid_top10_accuracy',
    'best_test_at_best_valid_accuracy', 'best_test_at_best_valid_macro_f1', 'best_test_at_best_valid_top5_accuracy', 'best_test_at_best_valid_top10_accuracy',
    'best_valid_time_mae', 'best_valid_time_rmse', 'best_valid_time_median_ae',
    'best_test_at_best_valid_time_mae', 'best_test_at_best_valid_time_rmse', 'best_test_at_best_valid_time_median_ae',
]].agg(['mean', 'std'])
summary_compare


best_valid_full_ndcg@10     best_valid_full_hr@10     best_valid_full_mrr     best_test_at_best_valid_full_ndcg@10     best_test_at_best_valid_full_hr@10     best_test_at_best_valid_full_mrr     best_valid_accuracy     best_valid_macro_f1     best_valid_top5_accuracy     best_valid_top10_accuracy     best_test_at_best_valid_accuracy     best_test_at_best_valid_macro_f1     best_test_at_best_valid_top5_accuracy     best_test_at_best_valid_top10_accuracy     best_valid_time_mae     best_valid_time_rmse     best_valid_time_median_ae     best_test_at_best_valid_time_mae     best_test_at_best_valid_time_rmse     best_test_at_best_valid_time_median_ae    
                                      mean std                  mean std                mean std                                 mean std                               mean std                             mean std                       mean std                       mean std                            mean std                             mean std                                    mean std                                    mean std                                         mean std                                          mean std                       mean std                        mean std                             mean std                                    mean std                                     mean std                                          mean std
variant                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               
anchor_multi_task                 0.742361 NaN              0.976065 NaN            0.671685 NaN                             0.809234 NaN                                1.0 NaN                         0.745731 NaN                   0.053685 NaN                   0.052223 NaN                        0.408384 NaN                         0.692765 NaN                                0.014680 NaN                                0.011972 NaN                                     0.478864 NaN                                      0.668207 NaN               74664.950418 NaN               282278.030256 NaN                      6863.672687 NaN                            14168.895027 NaN                             78819.227077 NaN                                    129.408642 NaN
anchor_single_task                0.750367 NaN              0.977546 NaN            0.681799 NaN                             0.915296 NaN                                1.0 NaN                         0.886842 NaN                        NaN NaN                        NaN NaN                             NaN NaN                              NaN NaN                                     NaN NaN                                     NaN NaN                                          NaN NaN                                           NaN NaN                        NaN NaN                         NaN NaN                              NaN NaN                                     NaN NaN                                      NaN NaN                                           NaN NaN
refine_multi_task                 0.733884 NaN              0.968635 NaN            0.668363 NaN                             0.661596 NaN                                1.0 NaN                         0.551116 NaN                   0.063544 N

Interpretation guide:
- compare `anchor_single_task` vs `anchor_multi_task` first
- compare `refine_single_task` vs `refine_multi_task` next
- use `best_test_at_best_valid_full_ndcg@10` as the main Stage 3 comparison metric
- only run additional seeds if the initial `s42` results are promising
